# 03 - Rolling-window training

Step 3 of `THESIS_GUIDE.md`. The six models are trained on the FinBench rolling protocol (4 years of training, 1 of validation, 1 of test; test years 2020-2024) in three universes (DJI, NASDAQ-100, EURO STOXX 50), with T = 20 and L = 5. Each model is trained **exactly as FinBench trains it** and keeps its weights with FinBench's own rule (decided 2026-10-06, `xaifin.training.trainer.SELECTION_RULES`); the data-quality filter is on.

This notebook reads the results store (`xaifin.training.rolling.collect_results`) and answers three questions:

1. **How well do the models predict?** IC, RankIC, ICIR and MSE per model, universe and test year.
2. **Are the numbers in FinBench's range?** FinBench publishes only MSE and MAE for these models (`results/analysis/finbench_table2.csv`).
3. **How weak is the signal?** The starting point of the thesis: if the predictions barely rank the stocks, explanations and portfolios built on them can be unstable.

Metrics are computed on the test year with the labels the model is scored on (the daily z-score of the forward return). **IC** and **RankIC** are the daily cross-sectional Pearson and Spearman correlations between predictions and labels, averaged over the test days; **ICIR** and **RankICIR** divide that mean by the standard deviation of the daily values. FactorVAE's label is a 20-day return (issue 3 in `docs/model_notes.md`), and FinFormer's loss ignores scale, so its MSE is not comparable (notebook 02).

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from xaifin import viz
from xaifin.config import CORE_UNIVERSES, RESULTS_ROOT, TEST_YEARS, UNIVERSE_NAMES
from xaifin.training.rolling import collect_results

SEQ_LEN, PRED_LEN = 20, 5
MODELS = viz.MODEL_ORDER
all_runs = collect_results()
runs = all_runs[(all_runs["seq_len"] == SEQ_LEN) & (all_runs["pred_len"] == PRED_LEN) & all_runs["universe"].isin(CORE_UNIVERSES)]
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print(f"{len(runs)} finished runs (T={SEQ_LEN}, L={PRED_LEN}), seeds {sorted(runs['seed'].unique())}")

15 finished runs (T=20, L=5), seeds [np.int64(42)]


## 1. The grid

One run per model, universe, test year and seed. EURO STOXX 50 has no 2020 run: its membership history starts on 2020-09-30, so no stock is a member at the start of 2020 (notebook 01). Training time is on one RTX 5090; FactorVAE is the slow one (FinBench's 96 attention layers run one after another for every day).

In [2]:
expected = {u: [y for y in TEST_YEARS if not (u == "sx5e" and y == 2020)] for u in CORE_UNIVERSES}
coverage = (runs.groupby(["model", "universe"]).size().unstack("universe").reindex(index=MODELS, columns=CORE_UNIVERSES)
            .fillna(0).astype(int))
coverage.loc["expected per model"] = [len(expected[u]) * runs["seed"].nunique() for u in CORE_UNIVERSES]
display(coverage)
timing = runs.groupby("model").agg(runs=("minutes", "size"), minutes_mean=("minutes", "mean"), minutes_total=("minutes", "sum"),
                                   epochs_run=("epochs_run", "mean"), kept_epoch=("kept_epoch", "mean")).reindex(MODELS)
timing

universe,dji,nasdaq100,sx5e
model,,,
MASTER,5,5,0
MATCC,1,0,0
FactorVAE,1,0,0
HIST,1,0,0
DiscoverPLF,1,0,0
FinFormer,1,0,0
expected per model,5,5,4


,runs,minutes_mean,minutes_total,epochs_run,kept_epoch
model,,,,,
MASTER,10,1.8300,18.3000,11.5000,10.5000
MATCC,1,12.6000,12.6000,70.0000,69.0000
FactorVAE,1,49.9000,49.9000,30.0000,1.0000
HIST,1,5.8000,5.8000,39.0000,8.0000
DiscoverPLF,1,17.0000,17.0000,49.0000,23.0000
FinFormer,1,2.8000,2.8000,16.0000,5.0000


## 2. Test performance per model, universe and year

Mean over the seeds trained so far. Blue cells: the predictions rank the stocks in the right direction on average over the test year; red: the wrong direction.

In [3]:
cells = runs.groupby(["universe", "model", "test_year"])[["IC", "RankIC", "ICIR", "RankICIR", "MSE", "MAE"]].mean().reset_index()
for universe in CORE_UNIVERSES:
    grid = cells[cells["universe"] == universe]
    if grid.empty:
        continue
    viz.metric_heatmap(grid, "RankIC", f"{UNIVERSE_NAMES[universe]}: test RankIC by model and test year").show()
    display(grid.pivot(index="model", columns="test_year", values="RankIC").reindex([m for m in MODELS if m in set(grid["model"])]))

test_year,2020,2021,2022,2023,2024
model,,,,,
MASTER,0.0529,-0.0109,-0.0002,0.0060,0.0201
MATCC,0.0248,NaN,NaN,NaN,NaN
FactorVAE,0.0501,NaN,NaN,NaN,NaN
HIST,0.0365,NaN,NaN,NaN,NaN
DiscoverPLF,0.0087,NaN,NaN,NaN,NaN
FinFormer,0.0301,NaN,NaN,NaN,NaN


test_year,2020,2021,2022,2023,2024
model,,,,,
MASTER,0.0390,0.0110,0.0102,0.0070,0.0091


In [4]:
for universe in CORE_UNIVERSES:
    grid = cells[cells["universe"] == universe]
    if not grid.empty:
        viz.metric_by_year_chart(grid, "IC", f"{UNIVERSE_NAMES[universe]}: test IC per year").show()

## 3. Averages over the test years, and FinBench's numbers

Per model and universe: the mean over test years (and seeds) of each metric, and its standard deviation across years. FinBench's Table 2 reports MSE and MAE averaged over its rolling test years and seeds; its FactorVAE numbers come from the leaky readout (issue 1), and its MASTER, MATCC and FinFormer runs were not seeded (issue 8).

In [5]:
finbench = pd.read_csv(RESULTS_ROOT / "analysis" / "finbench_table2.csv").query("pred_len == @PRED_LEN")
summary = (runs.groupby(["universe", "model"])
           .agg(years=("test_year", "nunique"), IC=("IC", "mean"), IC_std=("IC", "std"), RankIC=("RankIC", "mean"),
                RankIC_std=("RankIC", "std"), ICIR=("ICIR", "mean"), RankICIR=("RankICIR", "mean"),
                MSE=("MSE", "mean"), MAE=("MAE", "mean"))
           .reset_index()
           .merge(finbench.rename(columns={"MSE": "FinBench MSE", "MAE": "FinBench MAE"})[["universe", "model", "FinBench MSE", "FinBench MAE"]],
                  on=["universe", "model"], how="left"))
summary["model"] = pd.Categorical(summary["model"], MODELS)
summary = summary.sort_values(["universe", "model"]).reset_index(drop=True)
summary

,universe,model,years,IC,IC_std,RankIC,RankIC_std,ICIR,RankICIR,MSE,MAE,FinBench MSE,FinBench MAE
0,dji,MASTER,5,0.0158,0.0199,0.0136,0.0247,0.0713,0.0608,0.9806,0.7462,1.2490,0.8590
1,dji,MATCC,1,0.0069,NaN,0.0248,NaN,0.0263,0.0952,1.6015,0.9800,1.1300,0.8180
2,dji,FactorVAE,1,0.0123,NaN,0.0501,NaN,0.0505,0.2144,0.8532,0.6950,0.9710,0.7580
3,dji,HIST,1,0.0376,NaN,0.0365,NaN,0.1568,0.1471,0.9632,0.7453,0.9970,0.7470
4,dji,DiscoverPLF,1,-0.0073,NaN,0.0087,NaN,-0.0296,0.0347,1.1764,0.8276,1.2110,0.8300
5,dji,FinFormer,1,0.0253,NaN,0.0301,NaN,0.0883,0.1083,2.9259,1.3334,1.6570,0.9820
6,nasdaq100,MASTER,5,0.0111,0.0124,0.0152,0.0133,0.0575,0.0859,1.0139,0.7170,1.0170,0.7200


## 4. How weak is the signal?

Three views of the same question. (a) The share of runs whose test RankIC is above 0: a model with no skill lands there half of the time. (b) The t-statistic of the mean daily RankIC, RankICIR × √(test days): about 2 or more means the year's mean is unlikely to be noise. (c) The best test RankIC per universe and year: the ceiling of what any of the six reaches.

In [6]:
days = 252
weak = runs.assign(t_stat=runs["RankICIR"] * np.sqrt(days))
share = (weak.groupby("model").agg(runs=("RankIC", "size"), positive_RankIC=("RankIC", lambda s: (s > 0).mean()),
                                   mean_RankIC=("RankIC", "mean"), significant=("t_stat", lambda s: (s.abs() >= 2).mean()))
         .reindex(MODELS))
display(share.style.format({"positive_RankIC": "{:.0%}", "significant": "{:.0%}", "mean_RankIC": "{:+.4f}"}))
best = weak.loc[weak.groupby(["universe", "test_year"])["RankIC"].idxmax(), ["universe", "test_year", "model", "RankIC", "t_stat"]]
best

,runs,positive_RankIC,mean_RankIC,significant
model,,,,
MASTER,10,80%,+0.0144,20%
MATCC,1,100%,+0.0248,0%
FactorVAE,1,100%,+0.0501,100%
HIST,1,100%,+0.0365,100%
DiscoverPLF,1,100%,+0.0087,0%
FinFormer,1,100%,+0.0301,0%


,universe,test_year,model,RankIC,t_stat
4,dji,2020,MASTER,0.0529,3.4307
5,dji,2021,MASTER,-0.0109,-0.7038
6,dji,2022,MASTER,-0.0002,-0.0170
7,dji,2023,MASTER,0.0060,0.4247
8,dji,2024,MASTER,0.0201,1.6874
9,nasdaq100,2020,MASTER,0.0390,3.4849
10,nasdaq100,2021,MASTER,0.0110,0.8847
11,nasdaq100,2022,MASTER,0.0102,0.6710
12,nasdaq100,2023,MASTER,0.0070,0.8131
13,nasdaq100,2024,MASTER,0.0091,0.9632


## 5. The selection rules

Each model keeps its weights with FinBench's rule. If the validation score that picks the weights carried information about the test year, the kept epoch's validation IC and the test IC would be related; with a weak signal they need not be. The table shows, per model, the correlation across runs between the validation IC of the kept epoch and the test IC, and how many epochs the rule ran and kept.

In [7]:
selection = (runs.groupby("model")
             .apply(lambda d: pd.Series({"runs": len(d), "valid_IC_mean": d["valid_IC"].mean(), "test_IC_mean": d["IC"].mean(),
                                         "corr(valid IC, test IC)": d["valid_IC"].corr(d["IC"]) if len(d) > 2 else np.nan,
                                         "epochs_run": d["epochs_run"].mean(), "kept_epoch": d["kept_epoch"].mean()}),
                    include_groups=False)
             .reindex(MODELS))
selection

,runs,valid_IC_mean,test_IC_mean,"corr(valid IC, test IC)",epochs_run,kept_epoch
model,,,,,,
MASTER,10.0000,0.0063,0.0135,-0.1025,11.5000,10.5000
MATCC,1.0000,-0.0031,0.0069,NaN,70.0000,69.0000
FactorVAE,1.0000,-0.0301,0.0123,NaN,30.0000,1.0000
HIST,1.0000,0.0190,0.0376,NaN,39.0000,8.0000
DiscoverPLF,1.0000,0.0159,-0.0073,NaN,49.0000,23.0000
FinFormer,1.0000,-0.0015,0.0253,NaN,16.0000,5.0000


## 6. Saved summary

`results/analysis/metrics_summary.csv`: one row per run (cell, test metrics, kept epoch, epochs run, minutes, validation IC of the kept epoch). Later steps and the thesis tables read it.

In [8]:
out = RESULTS_ROOT / "analysis" / "metrics_summary.csv"
all_runs.drop(columns="path").sort_values(["model", "universe", "seq_len", "pred_len", "test_year", "seed"]).to_csv(out, index=False)
print(f"{len(all_runs)} runs -> {out}")

15 runs -> D:\CodeProjects\master-thesis\code\results\analysis\metrics_summary.csv


## Observations

OBSERVATIONS_PLACEHOLDER